# CNN Retinal OCT Classification
Compare a dense network and two small CNNs on four-class, 28×28 OCTMNIST.
Evaluate predictive performance alongside parameter count and training time.

**Status:** implementation prepared; OCTMNIST experiments and results are pending.
Data: [MedMNIST](https://medmnist.com/) ·
[OCTMNIST metadata and source citation](https://github.com/MedMNIST/MedMNIST/blob/main/medmnist/info.py).
Research benchmark; not intended for clinical use.


## 1. Setup
Set execution switches below. Enable data loading before training; reserve test evaluation until the experiment settings are final.


In [ ]:
import hashlib
import importlib.metadata
import json
import platform
import random
import subprocess
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision import transforms
from medmnist import OCTMNIST, INFO
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                             confusion_matrix, f1_score, recall_score)

LOAD_DATA = False
RUN_SMOKE_TEST = False
RUN_TRAINING = False
RUN_TEST_EVALUATION = False

CONFIG = {
    'image_size': 28, 'num_classes': 4,
    'batch_size': 256, 'epochs': 20, 'lr': 0.001,
    'seeds': [42, 123, 456],
    'normalization': 'ToTensor then subtract 0.5',
    'checkpoint_rule': 'minimum unweighted validation cross entropy',
    'model_version': 'dense-11-14-17-20-23-elu_oneconv-4-8_twoconv-4-8-8_v1',
}
RUN_NAME = 'baseline_v1'  # Change for a new experiment; existing runs are protected.
PROJECT = Path.cwd().resolve()  # Open the project folder in VS Code before running.
DATA_DIR = PROJECT / 'data'
RUN_DIR = PROJECT / 'results' / RUN_NAME
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CLASS_NAMES = ['CNV', 'DME', 'Drusen', 'Normal']
CONFIG_HASH = hashlib.sha256(json.dumps(CONFIG, sort_keys=True).encode()).hexdigest()

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

def synchronize():
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()

print('Python:', sys.executable)
print('PyTorch:', torch.__version__, '| Device:', DEVICE)
if DEVICE.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
print('Project:', PROJECT)


## 2. Load and inspect data
Use official training and validation splits, identical normalization, and shuffled training batches. Test predictions are reserved for Section 7.


In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.5,), std=(1.0,)),
])

def load_split(split):
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    dataset = OCTMNIST(split=split, root=str(DATA_DIR), size=28,
                      transform=transform, download=True)
    assert len(dataset) == INFO['octmnist']['n_samples'][split]
    return dataset

def make_loader(dataset, shuffle=False, seed=42):
    return DataLoader(dataset, batch_size=CONFIG['batch_size'],
                      shuffle=shuffle, drop_last=False, num_workers=0,
                      generator=torch.Generator().manual_seed(seed),
                      pin_memory=DEVICE.type == 'cuda')

if LOAD_DATA:
    train_ds, val_ds = load_split('train'), load_split('val')
    rows = []
    for split, dataset in [('train', train_ds), ('val', val_ds)]:
        counts = np.bincount(dataset.labels.reshape(-1), minlength=4)
        rows.extend({'split': split, 'class': name, 'count': int(count),
                     'fraction': float(count / len(dataset))}
                    for name, count in zip(CLASS_NAMES, counts))
    class_counts = pd.DataFrame(rows)
    display(class_counts)
    x, y = next(iter(make_loader(train_ds)))
    assert x.ndim == 4 and tuple(x.shape[1:]) == (1, 28, 28)
    assert x.dtype == torch.float32 and torch.isfinite(x).all()
    assert x.min() >= -0.5 and x.max() <= 0.5
    assert y.reshape(-1).min() >= 0 and y.reshape(-1).max() < 4
    print('Image batch:', tuple(x.shape), '| Targets:', tuple(y.reshape(-1).shape))

    rng = np.random.default_rng(42)
    fig, axes = plt.subplots(4, 4, figsize=(8, 8))
    for label in range(4):
        indices = rng.choice(np.flatnonzero(train_ds.labels.reshape(-1) == label),
                             size=4, replace=False)
        for ax, index in zip(axes[label], indices):
            ax.imshow(train_ds.imgs[index], cmap='gray', vmin=0, vmax=255)
            ax.set_title(CLASS_NAMES[label])
            ax.axis('off')
    fig.tight_layout()
    plt.show()
else:
    print('Data loading skipped. Set LOAD_DATA=True to inspect OCTMNIST.')


## 3. Define models
Each model accepts `[batch, 1, 28, 28]` and returns four logits. Parameter counts differ, so this compares practical baselines rather than isolating the effect of depth.


In [ ]:
class DenseNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        widths = [784, 11, 14, 17, 20, 23]
        layers = [nn.Flatten()]
        for in_features, out_features in zip(widths[:-1], widths[1:]):
            layers.extend([nn.Linear(in_features, out_features), nn.ELU()])
        self.network = nn.Sequential(*layers, nn.Linear(widths[-1], 4))

    def forward(self, x):
        return self.network(x)

class OneConvNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Conv2d(1, 4, kernel_size=3, stride=2), nn.ReLU(),
            nn.Flatten(), nn.Linear(4 * 13 * 13, 8), nn.ReLU(), nn.Linear(8, 4))

    def forward(self, x):
        return self.network(x)

class TwoConvNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Conv2d(1, 4, kernel_size=5, stride=2), nn.ReLU(),
            nn.Conv2d(4, 8, kernel_size=3, stride=2), nn.ReLU(),
            nn.Flatten(), nn.Linear(8 * 5 * 5, 8), nn.ReLU(), nn.Linear(8, 4))

    def forward(self, x):
        return self.network(x)

MODEL_BUILDERS = {'Dense': DenseNetwork, 'OneConv': OneConvNetwork,
                  'TwoConv': TwoConvNetwork}

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

display(pd.DataFrame([{'model': name, 'parameters': count_parameters(build())}
                      for name, build in MODEL_BUILDERS.items()]))


## 4. Training and evaluation functions
Train with cross entropy and save the weights with the lowest validation loss. Report macro-F1, accuracy, balanced accuracy, and recall for every class.


In [ ]:
def classification_metrics(targets, predictions):
    recalls = recall_score(targets, predictions, labels=range(4),
                           average=None, zero_division=0)
    return {
        'accuracy': float(accuracy_score(targets, predictions)),
        'macro_f1': float(f1_score(targets, predictions, labels=range(4),
                                    average='macro', zero_division=0)),
        'balanced_accuracy': float(balanced_accuracy_score(targets, predictions)),
        **{f'recall_{name}': float(value) for name, value in zip(CLASS_NAMES, recalls)},
    }

def run_epoch(model, loader, optimizer=None, max_batches=None):
    training = optimizer is not None
    model.train(training)
    total_loss, total_samples = 0.0, 0
    all_targets, all_predictions, all_probabilities = [], [], []
    with torch.set_grad_enabled(training):
        for batch_index, (images, labels) in enumerate(loader):
            if max_batches is not None and batch_index >= max_batches:
                break
            images = images.to(DEVICE)
            labels = labels.reshape(-1).long().to(DEVICE)
            if training:
                optimizer.zero_grad(set_to_none=True)
            logits = model(images)
            loss = nn.functional.cross_entropy(logits, labels)
            if not torch.isfinite(loss):
                raise RuntimeError('Non-finite loss encountered.')
            if training:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * len(labels)
            total_samples += len(labels)
            all_targets.append(labels.detach().cpu().numpy())
            all_predictions.append(logits.detach().argmax(1).cpu().numpy())
            all_probabilities.append(logits.detach().softmax(1).cpu().numpy())
    if total_samples == 0:
        raise ValueError('The loader supplied no samples.')
    targets = np.concatenate(all_targets)
    predictions = np.concatenate(all_predictions)
    metrics = classification_metrics(targets, predictions)
    metrics['loss'] = total_loss / total_samples
    return metrics, targets, predictions, np.concatenate(all_probabilities)

def save_checkpoint(path, model, name, seed, epoch, val_loss):
    torch.save({'state_dict': {k: v.detach().cpu().clone()
                              for k, v in model.state_dict().items()},
                'model': name, 'seed': seed, 'epoch': epoch,
                'val_loss': float(val_loss), 'config': CONFIG,
                'config_hash': CONFIG_HASH}, path)

def load_checkpoint(path, name, seed):
    checkpoint = torch.load(path, map_location='cpu', weights_only=True)
    if (checkpoint['config_hash'] != CONFIG_HASH or checkpoint['model'] != name
            or checkpoint['seed'] != seed):
        raise ValueError('Checkpoint does not match the current experiment.')
    model = MODEL_BUILDERS[name]().to(DEVICE)
    model.load_state_dict(checkpoint['state_dict'])
    return model, checkpoint

def train_model(name, seed, train_dataset, val_dataset, directory):
    seed_everything(seed)
    model = MODEL_BUILDERS[name]().to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG['lr'])
    train_loader = make_loader(train_dataset, shuffle=True, seed=seed)
    val_loader = make_loader(val_dataset)
    checkpoint_path = directory / f'{name}_seed{seed}.pt'
    history, best_loss, best_epoch = [], float('inf'), None
    synchronize()
    start = time.perf_counter()
    for epoch in range(1, CONFIG['epochs'] + 1):
        train_metrics, *_ = run_epoch(model, train_loader, optimizer)
        val_metrics, *_ = run_epoch(model, val_loader)
        history.append({'epoch': epoch,
                        **{f'train_{k}': v for k, v in train_metrics.items()},
                        **{f'val_{k}': v for k, v in val_metrics.items()}})
        if val_metrics['loss'] < best_loss:
            best_loss, best_epoch = val_metrics['loss'], epoch
            save_checkpoint(checkpoint_path, model, name, seed, epoch, best_loss)
        pd.DataFrame(history).to_csv(directory / f'{name}_seed{seed}_history.csv', index=False)
        print(f'{name} seed={seed} epoch={epoch:02d} '
              f'train_loss={train_metrics["loss"]:.4f} '
              f'val_loss={val_metrics["loss"]:.4f} val_F1={val_metrics["macro_f1"]:.4f}', flush=True)
    synchronize()
    elapsed = time.perf_counter() - start
    best_model, _ = load_checkpoint(checkpoint_path, name, seed)
    metrics, *_ = run_epoch(best_model, val_loader)
    return {'model': name, 'seed': seed, 'parameters': count_parameters(best_model),
            'best_epoch': best_epoch, 'training_seconds': elapsed, **metrics}


## 5. Smoke test
Optionally check shapes, gradients, parameter updates, metrics, and checkpoint reload on synthetic images. These checks produce no OCT performance results.


In [ ]:
def smoke_test():
    import tempfile
    seed_everything(42)
    synthetic = TensorDataset(torch.rand(16, 1, 28, 28) - 0.5,
                              (torch.arange(16) % 4).reshape(-1, 1))
    loader = DataLoader(synthetic, batch_size=8)
    perfect = classification_metrics(np.arange(4), np.arange(4))
    assert perfect['accuracy'] == perfect['macro_f1'] == 1.0
    with tempfile.TemporaryDirectory(dir=PROJECT) as temp:
        for name, build in MODEL_BUILDERS.items():
            model = build().to(DEVICE)
            before = {k: v.detach().clone() for k, v in model.state_dict().items()}
            optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG['lr'])
            metrics, targets, predictions, probabilities = run_epoch(model, loader, optimizer)
            assert len(targets) == 16 and probabilities.shape == (16, 4)
            assert np.isfinite(metrics['loss']) and np.allclose(probabilities.sum(1), 1)
            assert any(not torch.equal(before[k], v) for k, v in model.state_dict().items())
            assert all(p.grad is not None and torch.isfinite(p.grad).all() for p in model.parameters())
            path = Path(temp) / 'check.pt'
            save_checkpoint(path, model, name, 42, 1, metrics['loss'])
            restored, _ = load_checkpoint(path, name, 42)
            model.eval()
            restored.eval()
            sample = synthetic.tensors[0][:4].to(DEVICE)
            with torch.no_grad():
                assert torch.allclose(model(sample), restored(sample))
            evaluation, *_ = run_epoch(restored, loader)
            assert np.isfinite(evaluation['loss'])
            print(name, 'passed')

if RUN_SMOKE_TEST:
    smoke_test()
else:
    print('Smoke test skipped.')


## 6. Run and compare development experiments
Use the same settings and seeds for each model. Select the architecture using mean validation macro-F1; retain every prespecified model for the final comparison.


In [ ]:
def run_experiments():
    if not LOAD_DATA:
        raise RuntimeError('Enable LOAD_DATA and rerun Section 2 first.')
    if RUN_DIR.exists():
        raise FileExistsError('Run directory already exists. Choose a new RUN_NAME to preserve previous runs.')
    RUN_DIR.mkdir(parents=True)
    versions = {name: importlib.metadata.version(name) for name in
                ['torch', 'torchvision', 'medmnist', 'numpy', 'pandas', 'matplotlib', 'scikit-learn']}
    manifest = {'config': CONFIG, 'config_hash': CONFIG_HASH,
                'python': platform.python_version(), 'packages': versions,
                'device': str(DEVICE),
                'gpu': torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else None,
                'dataset': 'OCTMNIST official 28x28 splits',
                'class_names': CLASS_NAMES,
                'majority_training_class': int(np.bincount(
                    train_ds.labels.reshape(-1), minlength=4).argmax())}
    (RUN_DIR / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
    frozen = subprocess.run([sys.executable, '-m', 'pip', 'freeze'],
                            capture_output=True, text=True, check=True)
    (RUN_DIR / 'requirements-lock.txt').write_text(frozen.stdout, encoding='utf-8')
    rows = []
    for name in MODEL_BUILDERS:
        for seed in CONFIG['seeds']:
            rows.append(train_model(name, seed, train_ds, val_ds, RUN_DIR))
            pd.DataFrame(rows).to_csv(RUN_DIR / 'validation_results.csv', index=False)
    results = pd.DataFrame(rows)
    summary = results.groupby('model').agg(
        macro_f1_mean=('macro_f1', 'mean'), macro_f1_std=('macro_f1', 'std'),
        accuracy_mean=('accuracy', 'mean'), parameters=('parameters', 'first'),
        training_seconds_mean=('training_seconds', 'mean'))
    summary = summary.sort_values('macro_f1_mean', ascending=False, kind='stable')
    summary.to_csv(RUN_DIR / 'validation_summary.csv')
    selection = {'architecture': summary.index[0],
                 'rule': 'highest mean validation macro-F1; ties follow MODEL_BUILDERS order',
                 'config_hash': CONFIG_HASH}
    (RUN_DIR / 'selection.json').write_text(json.dumps(selection, indent=2), encoding='utf-8')
    display(summary)

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    for ax, name in zip(axes, MODEL_BUILDERS):
        for seed in CONFIG['seeds']:
            history = pd.read_csv(RUN_DIR / f'{name}_seed{seed}_history.csv')
            ax.plot(history.epoch, history.train_loss, linestyle='--', alpha=0.6,
                    label=f'train {seed}')
            ax.plot(history.epoch, history.val_loss, label=f'val {seed}')
        ax.set(title=name, xlabel='Epoch', ylabel='Cross entropy')
        ax.legend(fontsize=7)
    fig.tight_layout()
    fig.savefig(RUN_DIR / 'learning_curves.png', dpi=180)
    plt.show()

if RUN_TRAINING:
    run_experiments()
else:
    print('Training skipped. No OCTMNIST performance results have been generated by this cell.')


## 7. Final test evaluation and error analysis
After settings are frozen, evaluate the saved checkpoints once and save predictions. Do not retune using test results. Seed variation describes training variability, not clinical uncertainty.


In [ ]:
def final_test_evaluation():
    selection = json.loads((RUN_DIR / 'selection.json').read_text(encoding='utf-8'))
    if selection['config_hash'] != CONFIG_HASH:
        raise ValueError('Current configuration differs from the completed development run.')
    expected = [(name, seed) for name in MODEL_BUILDERS for seed in CONFIG['seeds']]
    # Check the full set before loading test images.
    for name, seed in expected:
        load_checkpoint(RUN_DIR / f'{name}_seed{seed}.pt', name, seed)
    test_dir = RUN_DIR / 'test'
    if test_dir.exists():
        raise FileExistsError('Test evaluation directory exists. Use saved predictions for further analysis.')
    test_ds = load_split('test')
    test_loader = make_loader(test_ds)
    test_dir.mkdir()
    rows = []
    for name, seed in expected:
        model, checkpoint = load_checkpoint(RUN_DIR / f'{name}_seed{seed}.pt', name, seed)
        metrics, targets, predictions, probabilities = run_epoch(model, test_loader)
        rows.append({'model': name, 'seed': seed, 'best_epoch': checkpoint['epoch'], **metrics})
        frame = pd.DataFrame({'index': np.arange(len(targets)), 'target': targets,
                              'prediction': predictions})
        for label, class_name in enumerate(CLASS_NAMES):
            frame[f'p_{class_name}'] = probabilities[:, label]
        frame.to_csv(test_dir / f'{name}_seed{seed}_predictions.csv', index=False)

        matrix = confusion_matrix(targets, predictions, labels=range(4))
        normalized = matrix / np.maximum(matrix.sum(axis=1, keepdims=True), 1)
        fig, axes = plt.subplots(1, 2, figsize=(10, 4))
        for ax, values, title in zip(axes, [matrix, normalized], ['Count', 'Recall by true class']):
            im = ax.imshow(values, cmap='Blues')
            ax.set(xticks=range(4), yticks=range(4), xticklabels=CLASS_NAMES,
                   yticklabels=CLASS_NAMES, xlabel='Predicted', ylabel='True', title=title)
            for i in range(4):
                for j in range(4):
                    ax.text(j, i, f'{values[i,j]:.2f}' if title != 'Count' else str(values[i,j]),
                            ha='center', va='center', color='red', fontsize=9)
            fig.colorbar(im, ax=ax)
        fig.suptitle(f'{name}, seed {seed}')
        fig.tight_layout()
        fig.savefig(test_dir / f'{name}_seed{seed}_confusion.png', dpi=180)
        plt.close(fig)
    results = pd.DataFrame(rows)
    manifest = json.loads((RUN_DIR / 'manifest.json').read_text(encoding='utf-8'))
    majority = manifest['majority_training_class']
    baseline = classification_metrics(test_ds.labels.reshape(-1),
                                      np.full(len(test_ds), majority))
    (test_dir / 'majority_baseline.json').write_text(
        json.dumps({'predicted_class': CLASS_NAMES[majority], **baseline}, indent=2), encoding='utf-8')
    results.to_csv(test_dir / 'test_results.csv', index=False)
    metrics_to_summarize = ['accuracy', 'macro_f1', 'balanced_accuracy'] + [f'recall_{c}' for c in CLASS_NAMES]
    summary = results.groupby('model')[metrics_to_summarize].agg(['mean', 'std'])
    summary.to_csv(test_dir / 'test_summary.csv')
    display(summary)
    print('Majority baseline:', baseline)
    print('Architecture selected on validation:', selection['architecture'])

    # Reproducible random sample of errors from the validation-selected model,
    # using the first prespecified seed (not the seed with the best test score).
    name, seed = selection['architecture'], CONFIG['seeds'][0]
    predictions = pd.read_csv(test_dir / f'{name}_seed{seed}_predictions.csv')
    errors = predictions[predictions.target != predictions.prediction]
    sample = errors.sample(n=min(12, len(errors)), random_state=42)
    sample.to_csv(test_dir / 'sampled_errors.csv', index=False)
    if len(sample):
        fig, axes = plt.subplots(3, 4, figsize=(11, 8))
        for ax in axes.flat:
            ax.axis('off')
        for ax, (_, row) in zip(axes.flat, sample.iterrows()):
            ax.imshow(test_ds.imgs[int(row['index'])], cmap='gray', vmin=0, vmax=255)
            ax.set_title(f'True: {CLASS_NAMES[int(row.target)]}\nPred: {CLASS_NAMES[int(row.prediction)]}')
        fig.tight_layout()
        fig.savefig(test_dir / 'sample_errors.png', dpi=180)
        plt.show()

if RUN_TEST_EVALUATION:
    final_test_evaluation()
else:
    print('Final test evaluation skipped.')


## 8. Findings
Results pending. After execution, summarize the performance–parameter tradeoff, class-specific failures, and variability across seeds.

Limitations: 28×28 images lose fine detail; image/label arrays alone do not verify patient independence. Performance on these splits does not establish generalization to new hospitals, devices, or patients. This experiment does not validate clinical use.
